# convT-as-flipped-padded-conv — worked example 2: Hand-checked 2x2 input with a 2x2 edge kernel

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `convT-as-flipped-padded-conv`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

On a tiny example the ConvT-as-flipped-padded-conv identity can be verified byte-exact (no tolerance). ConvT places a scaled copy of the kernel at every input position and sums the overlaps; the equivalent conv2d path pads by `K-1`, flips the kernel spatially, and swaps channel axes. Both must equal the matrix you get by summing the four kernel placements by hand.

## Worked solution

We use `x = [[1, 2], [3, 4]]` of shape `(1, 1, 2, 2)` and a kernel `w = [[1, 1], [0, 0]]` of shape `(1, 1, 2, 2)` (ConvT layout `(IC=1, OC=1, KH=2, KW=2)`).

**Step 1 - compute the ConvT output by hand.** Transposed conv scatters: each input value multiplies the whole kernel and is added into the output at an offset equal to that input's position. The output is `(1, 1, 3, 3)`. Input `1` at (0,0) deposits `[[1,1],[0,0]]` at rows 0-1, cols 0-1. Input `2` at (0,1) deposits `[[2,2],[0,0]]` at rows 0-1, cols 1-2. Input `3` at (1,0) deposits `[[3,3],[0,0]]` at rows 1-2, cols 0-1. Input `4` at (1,1) deposits `[[4,4],[0,0]]` at rows 1-2, cols 1-2. Summing the overlaps gives `[[1,3,2],[3,7,4],[0,0,0]]`.

**Step 2 - build the conv2d equivalent.** Pad `x` by `K-1 = 1` on each side. Flip the kernel spatially with `w.flip([2, 3])` and swap channel axes with `.transpose(0, 1)`. Run `F.conv2d`.

**Step 3 - assert exact equality.** Because all values are small integers represented exactly in float32, we use `t.equal` (not `allclose`) to confirm `convT_out`, the conv2d path, and the hand-computed `expected` matrix all agree exactly.

The lesson: the flip is not cosmetic. With a non-symmetric kernel like `[[1,1],[0,0]]`, omitting the flip would scatter the taps in the wrong direction and the hand-check would fail.

In [ ]:
import torch.nn.functional as F

x = t.tensor([[[[1., 2.], [3., 4.]]]])
w = t.tensor([[[[1., 1.], [0., 0.]]]])     # ConvT layout (IC=1, OC=1, KH=2, KW=2)

convT_out = F.conv_transpose2d(x, w)

x_pad = F.pad(x, (1, 1, 1, 1))
w_eq = w.flip([2, 3]).transpose(0, 1).contiguous()
conv_equiv = F.conv2d(x_pad, w_eq)

expected = t.tensor([[[[1., 3., 2.],
                       [3., 7., 4.],
                       [0., 0., 0.]]]])

print('convT:\n', convT_out[0, 0])
print('exact match convT == expected:', t.equal(convT_out, expected))
print('exact match conv_equiv == expected:', t.equal(conv_equiv, expected))